In [ ]:
import cdsapi
import time
import numpy as np
import pandas as pd

# this script is resumable for large downloads
# but you need to setup your cdsapi key first
# nano $HOME/.cdsapirc or nano ~/.cdsapirc

In [ ]:
import os
from pathlib import Path
# Get the path of the parent folder
from nocitis.download.cams_download import (
    process_and_save_csvs, 
    check_hourly_time_range_consistency,
    merge_cams_folder,engineer_features,
    build_grid_mapping,csv_to_npz_grid)


# Auto-locate project root containing the 'data' folder
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "data").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent

# Change working directory to NOCITIS/ root
os.chdir(ROOT_DIR)
print("Updated Working Directory:", os.getcwd())

# create your own ADS account:
url: https://ads.atmosphere.copernicus.eu/api
key: get your key

In [6]:
# provide target grid points
vec = pd.read_csv("data/grid/cams_lon_lat_vectors.csv")
lon_sub2 = vec["longitude_50"].values
lat_sub2 = vec["latitude_50"].values

In [7]:
# ---------------- 1. Generate grid over Hong Kong ----------------
lats = lat_sub2
lons = lon_sub2
points = [(round(lon, 4), round(lat, 4)) for lat in lats for lon in lons]

# ---------------- 2.  Constant request parts -------------------------
dataset   = "cams-solar-radiation-timeseries"
altitude  = ["0"]                                  # list!
date_span = ["2025-01-01/2025-01-02"]                  # list! ["2020-01-01/2025-06-19"]     
time_step = "15minute"
time_ref  = "universal_time"
sky_type  = "observed_cloud"                           # ALL-sky
fmt       = "csv"                               

# ---------------- 3.  ADS client and output folder -------------------
client  = cdsapi.Client()                              # uses ~/.cdsapirc
outdir  = "Data/CAMS/raw/2025/"  # output folder
os.makedirs(outdir, exist_ok=True)

# ---------------- 4.  Loop over every grid point ---------------------
for lon, lat in points:
    target = f"{outdir}/{lon}_{lat}.csv"
    if os.path.exists(target):
        continue                                       # skip if done
    req = {
        "sky_type"      : sky_type,
        "location"      : {"longitude": lon, "latitude": lat},
        "altitude"      : altitude,
        "date"          : date_span,
        "time_step"     : time_step,
        "time_reference": time_ref,
        "format"        : fmt
    }
    try:
        print(f"→ {lon},{lat}")
        client.retrieve(dataset, req).download(target)
        # time.sleep(1)                                
    except Exception as e:
        print(f"✖ {lon},{lat}  {e}")
# 2500 points

→ 113.64,22.86


2026-08-31 10:44:20,571 INFO Request ID is 3c62cf59-896d-4d42-a11c-33faf92d8908
2026-08-31 10:44:20,881 INFO status has been updated to accepted
2026-08-31 10:44:44,429 INFO status has been updated to running
2026-08-31 10:44:56,179 INFO status has been updated to successful


→ 113.66,22.86


2026-08-31 10:45:00,787 INFO Request ID is e7ac65fd-e116-4c27-8e70-40b77d51eb77
2026-08-31 10:45:01,079 INFO status has been updated to accepted


KeyboardInterrupt: 

In [8]:
process_and_save_csvs(
    source_root='Data/CAMS/raw/2025',
    target_root='Data/CAMS/raw/2025_15mins',
    header_row=42, # no. depends on the actual file header
    mode='instant',
)

✅ INSTANT saved: Data/CAMS/raw/2025_15mins/./113.64_22.86_instant.csv


In [9]:
valid, mismatches = check_hourly_time_range_consistency('Data/CAMS/2021-2024_15mins')

print(f"✅ {len(valid)} files have consistent time ranges.")

if mismatches:
    print(f"❌ {len(mismatches)} inconsistent files found:")
    for path, start, end, reason in mismatches:
        print(f" - {path}: {start} → {end} | {reason}")

✅ 0 files have consistent time ranges.


In [ ]:
merged = merge_cams_folder("Data/CAMS/2021-2024_15mins", "merged_2021-2024_15mins.csv")
print(merged.head())

In [ ]:
df, final_features = engineer_features(merged)

In [ ]:
df, grid_shape = build_grid_mapping(merged)
df_f = df[final_features + ['row', 'col']].copy()

In [ ]:
merged = merged['Observation period','Clear sky GHI','GHI',]

In [ ]:
data, time_arr = csv_to_npz_grid(
    df=df_f,
    out_path="ghi_grid2500_2ch_15mins.npz",
    grid_shape=(50, 50)
)